# MNIST Handwritten Digit Classification — CNN with PyTorch

**Dataset:** MNIST — 70,000 grayscale images of handwritten digits (0–9), 28×28 pixels  
**Model:** Convolutional Neural Network (CNN)  
**Framework:** PyTorch  


---
### 📌 Steps in this notebook
1. Setup & imports
2. Load & explore MNIST dataset
3. Define CNN architecture
4. Train the model
5. Evaluate — accuracy, confusion matrix, classification report
6. Visualise — training curves & sample predictions


## 1️⃣ Setup & Imports

In [ ]:
# Install any missing packages (torchvision is pre-installed on Colab)
!pip install -q seaborn scikit-learn

import os
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix

# ── Device ──
device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

torch.manual_seed(42)
print('Setup complete!')

## 2️⃣ Load & Explore the MNIST Dataset

In [ ]:
# ── Hyperparameters ──
BATCH_SIZE = 64
EPOCHS     = 5        # 5 epochs → ~99% accuracy in < 2 min on GPU!
LR         = 1e-3

# ── Transforms: convert to tensor & normalise with MNIST mean/std ──
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# ── Download MNIST ──
train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Training samples : {len(train_dataset):,}')
print(f'Test samples     : {len(test_dataset):,}')
print(f'Image shape      : {train_dataset[0][0].shape}')
print(f'Classes          : {train_dataset.classes}')

In [ ]:
# ── Visualise sample images ──
fig, axes = plt.subplots(3, 10, figsize=(15, 5))
fig.suptitle('MNIST Sample Images — One per Digit Class', fontsize=14, fontweight='bold')

for digit in range(10):
    indices = (train_dataset.targets == digit).nonzero(as_tuple=True)[0][:3]
    for row, idx in enumerate(indices):
        ax = axes[row, digit]
        ax.imshow(train_dataset[idx][0].squeeze(), cmap='gray')
        ax.axis('off')
        if row == 0:
            ax.set_title(str(digit), fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sample images saved!')

In [ ]:
# ── Class distribution ──
labels = train_dataset.targets.numpy()
counts = np.bincount(labels)

plt.figure(figsize=(10, 4))
bars = plt.bar(range(10), counts, color=plt.cm.tab10(np.linspace(0, 1, 10)), edgecolor='black')
plt.xlabel('Digit Class', fontsize=12)
plt.ylabel('Number of Samples', fontsize=12)
plt.title('MNIST Training Set — Class Distribution', fontsize=14, fontweight='bold')
plt.xticks(range(10))
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
             f'{count:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3️⃣ CNN Architecture

```
Input  (1 × 28 × 28)  — grayscale image
  │
  ▼
Conv2d(1→32, 3×3) + ReLU      →  32 × 26 × 26
  │
  ▼
Conv2d(32→64, 3×3) + ReLU     →  64 × 24 × 24
  │
  ▼
MaxPool2d(2×2) + Dropout(25%) →  64 × 12 × 12
  │
  ▼
Flatten                        →  9216
  │
  ▼
Linear(9216→128) + ReLU + Dropout(50%)
  │
  ▼
Linear(128→10)                 →  class logits
```

In [ ]:
class MnistCNN(nn.Module):
    """
    A simple but powerful CNN for MNIST digit classification.
    ~1.2 million parameters. Achieves ~99% test accuracy in 5 epochs.
    """
    def __init__(self):
        super(MnistCNN, self).__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(in_channels=1,  out_channels=32, kernel_size=3)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
        # Regularisation
        self.dropout1 = nn.Dropout(p=0.25)
        self.dropout2 = nn.Dropout(p=0.50)
        # Fully-connected layers
        self.fc1 = nn.Linear(64 * 12 * 12, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))           # (B, 32, 26, 26)
        x = F.relu(self.conv2(x))           # (B, 64, 24, 24)
        x = F.max_pool2d(x, kernel_size=2)  # (B, 64, 12, 12)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)             # (B, 9216)
        x = F.relu(self.fc1(x))             # (B, 128)
        x = self.dropout2(x)
        x = self.fc2(x)                     # (B, 10)
        return x


model = MnistCNN().to(device)

# ── Count parameters ──
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f'\nTotal parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable:,}')

## 4️⃣ Train the Model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = StepLR(optimizer, step_size=2, gamma=0.5)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
os.makedirs('checkpoints', exist_ok=True)

print(f'Starting training for {EPOCHS} epochs on {device}...')
print('=' * 75)
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>8} | {"Time":>6}')
print('-' * 75)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # ── Train ──
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss    += loss.item() * images.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total   += labels.size(0)

    # ── Validate ──
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss    += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total   += labels.size(0)

    scheduler.step()

    tl = train_loss / train_total
    ta = 100.0 * train_correct / train_total
    vl = val_loss / val_total
    va = 100.0 * val_correct / val_total
    elapsed = time.time() - t0

    history['train_loss'].append(tl)
    history['train_acc'].append(ta)
    history['val_loss'].append(vl)
    history['val_acc'].append(va)

    marker = ' <-- best!' if va > best_val_acc else ''
    print(f'{epoch:>6} | {tl:>10.4f} | {ta:>8.2f}% | {vl:>8.4f} | {va:>7.2f}%  | {elapsed:>5.1f}s{marker}')

    if va > best_val_acc:
        best_val_acc = va
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_acc': va}, 'checkpoints/best_model.pth')

print('=' * 75)
print(f'Training complete!  Best Validation Accuracy: {best_val_acc:.2f}%')

## 5️⃣ Evaluate the Model

In [ ]:
# ── Load best checkpoint & run full inference ──
ckpt = torch.load('checkpoints/best_model.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

all_preds, all_labels, all_probs, all_images = [], [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        probs  = torch.softmax(logits, dim=1)
        preds  = probs.argmax(dim=1)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.numpy())
        all_probs.append(probs.cpu().numpy())
        all_images.append(images.cpu().numpy())

y_pred  = np.concatenate(all_preds)
y_true  = np.concatenate(all_labels)
y_prob  = np.concatenate(all_probs)
images  = np.concatenate(all_images)

overall_acc = 100.0 * (y_pred == y_true).sum() / len(y_true)
print(f'Overall Test Accuracy : {overall_acc:.2f}%')
print(f'Misclassified images  : {(y_pred != y_true).sum()} / {len(y_true)}')

In [ ]:
# ── Classification Report ──
report = classification_report(y_true, y_pred, target_names=[str(i) for i in range(10)])
print('Classification Report')
print('=' * 55)
print(report)

with open('classification_report.txt', 'w') as f:
    f.write('MNIST CNN — Classification Report\n')
    f.write('=' * 45 + '\n')
    f.write(report)
print('Saved to classification_report.txt')

In [ ]:
# ── Confusion Matrix ──
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10))
plt.xlabel('Predicted Label', fontsize=13)
plt.ylabel('True Label', fontsize=13)
plt.title('Confusion Matrix — MNIST CNN', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved confusion_matrix.png')

## 6️⃣ Visualise Training Curves & Sample Predictions

In [ ]:
# ── Training Curves ──
epochs_range = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs_range, history['train_loss'], 'o-', label='Train Loss', color='#2196F3', linewidth=2)
ax1.plot(epochs_range, history['val_loss'],   's--', label='Val Loss',  color='#FF5722', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12); ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Loss per Epoch', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11); ax1.grid(alpha=0.3)

ax2.plot(epochs_range, history['train_acc'], 'o-', label='Train Acc', color='#4CAF50', linewidth=2)
ax2.plot(epochs_range, history['val_acc'],   's--', label='Val Acc',  color='#9C27B0', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12); ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Accuracy per Epoch', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11); ax2.grid(alpha=0.3)

fig.suptitle('MNIST CNN — Training Curves', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

In [ ]:
# ── Sample Predictions Grid (6×6) ──
wrong_idx = np.where(y_pred != y_true)[0][:9]
right_idx = np.where(y_pred == y_true)[0][:27]
idx = np.concatenate([right_idx, wrong_idx])[:36]

fig = plt.figure(figsize=(14, 14))
gs  = gridspec.GridSpec(6, 6, figure=fig, hspace=0.5, wspace=0.3)
fig.suptitle('Sample Predictions  (green = correct  |  red = wrong)', fontsize=14, fontweight='bold')

for pos, i in enumerate(idx):
    ax = fig.add_subplot(gs[pos // 6, pos % 6])
    ax.imshow(images[i, 0], cmap='gray')
    ax.axis('off')
    correct = y_pred[i] == y_true[i]
    conf    = y_prob[i, y_pred[i]] * 100
    color   = '#2e7d32' if correct else '#c62828'
    ax.set_title(f'P:{y_pred[i]}  T:{y_true[i]}\n{conf:.1f}%', fontsize=8, color=color)

plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved sample_predictions.png')

In [ ]:
# ── Final Summary ──
print('=' * 50)
print('  MNIST CNN — Final Results Summary')
print('=' * 50)
print(f'  Epochs trained       : {EPOCHS}')
print(f'  Best Val Accuracy    : {best_val_acc:.2f}%')
print(f'  Final Test Accuracy  : {overall_acc:.2f}%')
print(f'  Misclassified        : {(y_pred != y_true).sum()} / {len(y_true)}')
print(f'  Device used          : {device}')
print('=' * 50)
print()
print('Files saved in this session:')
for f in ['sample_images.png','class_distribution.png','training_curves.png',
          'confusion_matrix.png','sample_predictions.png','classification_report.txt']:
    if os.path.exists(f):
        print(f'  [OK] {f}')